In [2]:
import pandas as pd
import geopandas as gpd

In [3]:
# Setup
all_gdf = [] # Collect all datasets here

In [4]:
# Basel
df = pd.read_csv("../data/concentration_datasets/2026-09-23_Basel.csv", delimiter=";")

df_ = pd.read_csv("../data/concentration_datasets/2026-09-23_Basel-Rhein.csv", delimiter=";")

df = pd.concat([df, df_])

# Rename columns
cols = {
    'Probentyp' : "matrix", 
    'Probenahmestelle' : "name", 
    'X-Koordinate' : "lon",
    'Y-Koordinate' : "lat", 
    'Bestimmungsgrenze' : "bg", 
    'Wert numerisch' : 'conc', 
    'Einheit' : 'unit', 
    'CAS-Bezeichnung' : 'CAS', 
    "Parameter" : "parameter",
    'BAFU-Bezeichnung' : 'substance', 
    'Probenahmedatum_date' : "date", 
    }

df = df[cols.keys()]
df = df.rename(columns=cols)

df["less_than"] = False
df.loc[df["conc"].isna(), "less_than"] = True
df.loc[df["conc"].isna(), "conc"] = df.loc[df["conc"].isna(), "bg"]
df = df[df.unit != "ng_TEQ/L"]
df = df.drop_duplicates(subset=["substance", "conc", "lat", "lon", "date"])

cas_subst_rename = {
    "754-91-6" : "FOSA",
    "34455-29-3" : "6:2 FTAB",
    "80475-32-7" : "6:2 FTNO",
    "27619-97-2" : "6:2 FTS",
    "2991-50-6" : "EtFOSAA",
    "39108-34-4" : "8:2 FTS",
    "4151-50-2" : "N-Et-FOSA",
    "53826-13-4" : "10:2 FTCA",
    "120226-60-0" : "10:2 FTS",
    "53826-12-3" : "6:2 FTCA",
    "27854-31-5" : "8:2 FTCA",
    "70887-84-2" : "8:2 FTUCA",
    "757124-72-4" : "4:2 FTS",
    "67905-19-5" : "PFHxDA",
    "31506-32-8" : "N-MeFOSA",
    "2806-24-8" : "FOSAA",
    "16517-11-6" : "PFODA",
    "376-06-7" : "PFTeDA",
    "375-92-8" : "PFHpS",
    "79780-39-5" : "PFDoDS",
    "2058-94-8" : "PFUnDA",
    "307-55-1" : "PFDoDA"
}

for k, v in cas_subst_rename.items():
    df.loc[df.CAS == k, "substance"] = v

# df[["substance", "CAS", "parameter"]].drop_duplicates(subset=["substance", "CAS", "parameter"]).to_excel("basel_substances.xlsx", index=False)

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df.lon, df.lat, crs="ETRS89"), data=df
)
gdf["date"] = pd.to_datetime(gdf["date"], format="%Y-%m-%d")
gdf = gdf.to_crs("EPSG:4326")

gdf["source_type"] = "Authorities"
gdf["dataset_name"] = "_Basel_river"

all_gdf.append(gdf)

In [5]:
# ELWAS web 
df = pd.read_excel("../data/concentration_datasets/2026-10-01_Einzelwerte Chemie.xlsx", skiprows=3)

cols = {
    'Probengut' : "matrix", 
    'Messstellenname' : "name", 
    'Ostwert in UTM' : "lon",
    'Nordwert in UTM' : "lat", 
    'Messergebnis' : 'conc', 
    'Einheit' : 'unit', 
    'Stoffname' : 'substance', 
    'Datum' : "date", 
    "Gewässername" : "river"
    }

df = df[cols.keys()]
df = df.rename(columns=cols)

substance_rename = {
    'Perfluorbutansäure' : "PFBA",
    'Perfluorpentansäure' : "PFPeA",
    'Perfluorhexansäure' : "PFHxA",
    'Perfluorheptansäure' : "PFHpA",
    'Perfluornonansäure' : "PFNA",
    'Perfluordekansäure' : "PFDA",
    'Perfluorundekansäure' : "PFUnDA",
    'Perfluordekansulfonsäure' : "PFDS",
    'H4-Perfluoroktansulfonsäure' : "6:2 FTS",
    'H4-Perfluorhexansulfonsäure' : "4:2 FTS",
    'Perfluorheptansulfonsäure' : "PFHpS",
    'H4-Perfluordekansulfonsäure' : "8:2 FTS",
    'Perfluordodekansäure' : "PFDoDS",
    'Trifluoressigsaeure' : "TFA",
    'Perfluortridekansäure' : "PFTrDA",
    'Perfluoroktansäure' : "PFOA",
    'Perfluorbutansulfonsäure' : "PFBS",
    'Perfluorhexansulfonsäure' : "PFHxS",
    '7H-Perfluorheptansäure' : "HPFHpA",
    'Perfluorpentansulfonsäure' : "PFPeS",
    'HFPO-DA' : "HFPO-DA",
    'DONA' : "DONA",
    'Capstone A (DPOSA)' : "Capstone A",
    'Capstone B (CDPOS)' : "Capstone B",
    'Perfluoroctylsulfonsäureamid' : "FOSA",
}


df = df[df.substance.isin(substance_rename.keys())]
df["substance"] = df["substance"].replace(substance_rename)

df['less_than'] = False
df.loc[df.conc.str.contains(r"<"), 'less_than'] = True

df['conc'] = df.conc.str.replace("<","").str.split(" ").str[-1].str.replace(",",".").astype(float)
df = df.dropna()

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df.lon, df.lat, crs="EPSG:32632"), data=df
)

gdf = gdf.to_crs("EPSG:4326")
gdf["date"] = pd.to_datetime(gdf["date"], format="%d.%m.%Y")
gdf["source_type"] = "Authorities"
gdf["dataset_name"] = "_ELWAS"
all_gdf.append(gdf)

c:\Users\schroeder\Documents\Code\pfas_assessment_europe\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
import pandas as pd
import geopandas as gpd

df = pd.read_csv("../data/concentration_datasets/2026-09-23_Rijkswaterstaat.csv", delimiter=";")
cols = ['MEETPUNT_IDENTIFICATIE', 
       'PARAMETER_OMSCHRIJVING',
       'CAS_NR', 
       'EENHEID_CODE',
       'COMPARTIMENT_CODE',
       'WAARNEMINGDATUM', 
       'WAARNEMINGTIJD', 
       'LIMIETSYMBOOL', 
       'NUMERIEKEWAARDE',
       'EPSG', 
       'LAT', 
       'LON'
       ]

df_sel = df[cols]

df_sel = df_sel.rename(columns={
    'MEETPUNT_IDENTIFICATIE' : "name", 
    'PARAMETER_OMSCHRIJVING' : "parameter",
    'EENHEID_CODE' : "unit",
    'COMPARTIMENT_CODE' : "matrix",
    'WAARNEMINGDATUM' : "date", 
    'LIMIETSYMBOOL' : "less_than_symbol", 
    'NUMERIEKEWAARDE' : "conc",
    'LAT' : "lat", 
    'LON' : "lon",
    "CAS_NR" : "CAS"
})

substance_rename = {
    '6:2 fluortelomeersulfonzuur' : "6:2 FTS",
    "4,8-dioxa-3H-perfluornonaanzuur" : "DONA",
    "N-ethyl-perfluoroctaan sulfonamidoazijnzuur" : "EtFOSAA",
    "perfluor-2-propoxypropaanzuur" : "HFPO-DA",
    "8:2 fluortelomeersulfonzuur" : "8:2 FTS",
    "perfluor-1-octaansulfonaat (lineair)" : "L_PFOS",
    "N-methyl-perfluoroctaan sulfonamidoazijnzuur" : "N-MeFOSAA",
    "perfluorpentaansulfonzuur" : "PFPeS",
    "perfluornonaansulfonzuur" : "PFNS",
    "perfluoroctaansulfonamide" : "FOSA",
}

df_sel["substance"] = df_sel["parameter"].replace(substance_rename)

df_sel["less_than"] = False
df_sel.loc[df_sel["less_than_symbol"] == "<", "less_than"] = True

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df_sel.lon, df_sel.lat, crs="ETRS89"), data=df_sel
)
gdf["date"] = pd.to_datetime(gdf["date"], format="%d-%m-%Y")
gdf = gdf.to_crs("EPSG:4326")
gdf = gdf[gdf.substance.isin(substance_rename.values())].copy()
gdf["source_type"] = "Authorities"
gdf["dataset_name"] = "_Rijkswaterstaat_Waterinfo"

all_gdf.append(gdf)


gdf.dtypes

C:\Users\schroeder\AppData\Local\Temp\ipykernel_15028\2964756415.py:4: DtypeWarning: Columns (0: LIMIETSYMBOOL) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/concentration_datasets/2026-09-23_Rijkswaterstaat.csv", delimiter=";")


name                           str
parameter                      str
CAS                            str
unit                           str
matrix                         str
date                datetime64[us]
WAARNEMINGTIJD                 str
less_than_symbol               str
conc                       float64
EPSG                           str
lat                        float64
lon                        float64
substance                      str
less_than                     bool
geometry                  geometry
source_type                    str
dataset_name                   str
dtype: object

In [7]:
df = pd.read_excel("../data/concentration_datasets/ez4c01126_si_002.xlsx", sheet_name="Table S8", skiprows=1)

substance_cols = list(df.columns)
substance_cols = substance_cols[8:]

df = df[~df.latitude.isna()]

df["date"] = pd.to_datetime(df["Date of sampling"], format="mixed")

id_columns = ["Type of water", "latitude", "longitude", "date"]

df = df[substance_cols + id_columns]

df = df.melt(
    id_vars=id_columns,
    var_name = "substance",
    value_name="conc"
)

df["less_than"] = False
df.loc[df.conc == "nd", "less_than"] = True
df.loc[df.conc == "nd", "conc"] = 0.0
df.conc = df.conc.astype(float)
cols = {
    'Type of water' : "matrix", 
    'longitude' : "lon",
    'latitude' : "lat", 
    }

df = df.rename(columns=cols)

df = df[["matrix", "date", "lon", "lat", "substance", "conc", "less_than"]]

df = df[(df.matrix.str.startswith("River") | df.matrix.str.startswith("Pond"))]

df.matrix = "surface water"

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df.lon, df.lat, crs="EPSG:4326"), data=df
)
gdf["unit"] = "ng/l"
gdf["source_type"] = "Research"
gdf["dataset_name"] = "_teymoorian2025"
all_gdf.append(gdf)


C:\Users\schroeder\AppData\Local\Temp\ipykernel_15028\288157209.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["date"] = pd.to_datetime(df["Date of sampling"], format="mixed")


In [8]:
df.substance.unique()

<ArrowStringArray>
[         'PFPrA',           'PFBA',          'PFPeA',          'PFHxA',
          'PFHpA',           'PFOA',           'PFNA',           'PFDA',
         'PFUnDA',         'PFDoDA',
 ...
 'N-SPAmP-FPeSAA', 'N-SPAmP-FHxSAA', 'N-SPAmP-FHpSAA',  'N-SPAmP-FOSAA',
   'N-AmCP-FPrSA',    'N-AmCP-FBSA',   'N-AmCP-FPeSA',   'N-AmCP-FHxSA',
   'N-AmCP-FHpSA',    'N-AmCP-FOSA']
Length: 198, dtype: str

In [9]:
# Sweden 3 Streams https://doi.org/10.5878/hpqb-k059

# lon lat
coord_water ={
    "UppA":(17.578139, 59.886588)   ,
    "UppB1": (17.655366, 59.84268  ) ,
    "UppB2":( 17.656716, 59.841679 ) ,
    "UppC":(17.661265, 59.831679   ),
    "NrkA":(16.219937, 58.545871  ) ,
    "NrkB":(16.2217748, 58.5833263 ),
    "NrkC":(16.230462, 58.600521 )  ,
    "SdkA":(16.298764, 58.47634   ) ,
    "SdkB":(16.342321, 58.480029)   ,
    "SdkC":(16.381126, 58.478058)   
}
coord_sed = {
    "UppA" :(17.5771011, 59.8868957),
    "UppB1": (17.6554222, 59.8426312),
    "UppB2":  (17.6563436, 59.8417083),
    "UppC" :(17.6613627, 59.8316038),
    "NrkA": (16.3061581, 58.4765624),
    "NrkB" : (16.340546, 58.479974),
    "NrkC":  (16.3812429, 58.4780303),
    "SdkA": (16.2227602, 58.5505957),
    "SdkB": (16.2217748, 58.5833263),
    "SdkC": (16.229965, 58.600161)
}

df = pd.read_csv("../data/concentration_datasets/PFAS_in_water_34_targeted_PFAS.csv", sep=";")
df["unit"] = "ng/l"
sweden_dfs = []
subs = ['PFBA', 'PFPeA', 'PFHxA', 'PFHpA', 'PFOA', 'PFNA',
       'PFDA', 'PFBS', 'PFHxS', 'PFOS', '6:2 FTS', 'PFUnDA', 'PFDoDA',
       'PFTrDA', 'PFPeS ', 'PFHpS', 'PFNS ', 'PFDS', 'PFUnDS', 'PFDoDS',
       'PFTrDS', '4:2 FTS ', '8:2 FTS', 'FOSA', 'MeFOSA', 'EtFOSE', 'MeFOSE',
       'EtFOSA', 'FOSAA ', 'MeFOSAA', 'EtFOSAA', 'HPFHpA', 'PF37DMOA ',
       'PFTeDA']
for sub in subs:
    df_sub = df.copy()
    df_sub["substance"] = sub
    df_sub["conc"] = 0.0
    df_sub["less_than"] = False
    for i, row in df_sub.iterrows():
        conc_string: str = row[sub]
        if conc_string[0] == "<":
            split_conc_string = conc_string.split("<")
            df_sub.loc[i, "less_than"] = True
            df_sub.loc[i, "conc"] = float(split_conc_string[-1])
        else:
            df_sub.loc[i, "conc"] = float(conc_string)
    sweden_dfs.append(df_sub)
df_sweden = pd.concat(sweden_dfs)
df_sweden = df_sweden.drop(columns=subs)
df_sweden["matrix"] = "surface water"


df = pd.read_csv("../data/concentration_datasets/PFAS_ in_sediment_35_targeted_PFAS.csv", sep=";")
df = df.dropna(axis=0)
df["Event No"] = df["No"]
df = df.drop(columns="No")
df["unit"] = "µg/kg dw"
sweden_dfs = []
subs = ['PFBA', 'PFPeA',
       'PFHxA', 'PFHpA', 'PFOA', 'PFNA', 'PFDA', 'PFUnDA', 'PFDoDA', 'PFTeDA',
       'PFHxDA', 'HPFHpA', 'PF37DMOA', 'PFBS', 'PFHxS', 'PFHpS', 'PFOS',
       'PFDS', '4_2FTS', '6_2FTS', '8_2FTS', 'EtFOSA', 'EtFOSE', 'EtFOSAA',
       'MeFOSAA', 'MeFOSA', 'MeFOSE', 'FOSAA', 'PFNS', 'PFTrDA', 'PFDoDS',
       'PFPeS', 'PFTrDS', 'PFUnDS', 'FOSA']
for sub in subs:
    df_sub = df.copy()
    df_sub["substance"] = sub
    df_sub["conc"] = 0.0
    df_sub["less_than"] = False
    for i, row in df_sub.iterrows():
        conc_string: str = str(row[sub])
        if conc_string[0] == "<":
            split_conc_string = conc_string.split("<")
            df_sub.loc[i, "less_than"] = True
            df_sub.loc[i, "conc"] = float(split_conc_string[-1])
        else:
            df_sub.loc[i, "conc"] = float(conc_string)
    sweden_dfs.append(df_sub)
df_sweden_sed = pd.concat(sweden_dfs)
df_sweden_sed = df_sweden_sed.drop(columns=subs)
df_sweden_sed["matrix"] = "sediment"

df = pd.read_csv("../data/concentration_datasets/PFAS_in_sediment_30PFAS_TOP_Assay.csv", sep=";")
df = df.dropna(axis=0)
df["unit"] = "µg/kg dw"
df["Event No"] = df["No"]
df = df.drop(columns="No")
sweden_dfs = []
subs = ['4_2FTS', '6_2FTS',
       '8_2FTS', 'EtFOSA', 'EtFOSAA', 'EtFOSE', 'FOSAA', 'HPFHpA', 'MeFOSA',
       'MeFOSAA', 'MeFOSE', 'PF37DMOA', 'PFBA', 'PFBS', 'PFDA', 'PFDoDA',
       'PFDS', 'PFHpA', 'PFHpS', 'PFHxA', 'PFHxDA', 'PFHxS', 'PFNA', 'PFOA',
       'PFOS', 'FOSA', 'PFPeA', 'PFTeDA', 'PFTrDA', 'PFUnDA']
for sub in subs:
    df_sub = df.copy()
    df_sub["substance"] = sub
    df_sub["conc"] = 0.0
    df_sub["less_than"] = False
    for i, row in df_sub.iterrows():
        conc_string: str = str(row[sub])
        if conc_string[0] == "<":
            split_conc_string = conc_string.split("<")
            df_sub.loc[i, "less_than"] = True
            df_sub.loc[i, "conc"] = float(split_conc_string[-1])
        else:
            df_sub.loc[i, "conc"] = float(conc_string)
    sweden_dfs.append(df_sub)
df_sweden_top = pd.concat(sweden_dfs)
df_sweden_top = df_sweden_top.drop(columns=subs)
df_sweden_top["matrix"] = "sediment"

df_sweden_full = pd.concat([df_sweden, df_sweden_sed, df_sweden_top])

df_sweden_full["lon"] = 0.0
df_sweden_full["lat"] = 0.0

locations = df_sweden_full["Sample_Site"].unique()

for i, row in df_sweden_full.iterrows():
    site = row["Sample_Site"]
    if row["matrix"] == "surface water":
        lon = coord_water[site][0]
        lat = coord_water[site][1]
    elif row["matrix"] == "sediment":
        lon = coord_sed[site][0]
        lat = coord_sed[site][1]

    df_sweden_full.loc[i, "lon"] = lon
    df_sweden_full.loc[i, "lat"] = lat

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df_sweden_full.lon, df_sweden_full.lat, crs="EPSG:4326"), data=df_sweden_full
)
gdf["date"] = pd.to_datetime(gdf["Date"])
gdf["source_type"] = "Research"
gdf["dataset_name"] = "_kali2025"
print(gdf["substance"].unique())
gdf_sweden_streams = gdf.drop(columns="Date").copy()
gdf_sweden_streams = gdf_sweden_streams.rename(columns = {
    "Sample_Site" : "name",
})

all_gdf.append(gdf_sweden_streams)

<ArrowStringArray>
[     'PFBA',     'PFPeA',     'PFHxA',     'PFHpA',      'PFOA',      'PFNA',
      'PFDA',      'PFBS',     'PFHxS',      'PFOS',   '6:2 FTS',    'PFUnDA',
    'PFDoDA',    'PFTrDA',    'PFPeS ',     'PFHpS',     'PFNS ',      'PFDS',
    'PFUnDS',    'PFDoDS',    'PFTrDS',  '4:2 FTS ',   '8:2 FTS',      'FOSA',
    'MeFOSA',    'EtFOSE',    'MeFOSE',    'EtFOSA',    'FOSAA ',   'MeFOSAA',
   'EtFOSAA',    'HPFHpA', 'PF37DMOA ',    'PFTeDA',    'PFHxDA',  'PF37DMOA',
    '4_2FTS',    '6_2FTS',    '8_2FTS',     'FOSAA',      'PFNS',     'PFPeS']
Length: 42, dtype: str


In [10]:
gdf_merged = gpd.GeoDataFrame(
    pd.concat(all_gdf, ignore_index=True),
    crs=all_gdf[0].crs    # preserve CRS
)

cols_retain = ["matrix", "name", "lon", "lat", "conc", "unit", "substance", "date", "dataset_name", "source_type", "geometry", "less_than", "CAS"]

gdf_merged = gdf_merged[cols_retain]

assert gdf_merged.dtypes.less_than == bool
assert gdf_merged.dtypes.conc == float
assert gdf_merged.dtypes.date != object
assert gdf_merged.dtypes.date != str

gdf_merged.dtypes

matrix                     str
name                       str
lon                    float64
lat                    float64
conc                   float64
unit                       str
substance                  str
date            datetime64[ns]
dataset_name               str
source_type                str
geometry              geometry
less_than                 bool
CAS                        str
dtype: object

In [11]:
gdf_merged.substance.unique()

<ArrowStringArray>
[     'DONA',  'N-MeFOSA',    'PFDoDA',      'PFDS',     'PFHxA',     'PFHxS',
      'PFNA',      'PFOS',      'FOSA',     'PFPeA',
 ...
     'PFNS ',  '4:2 FTS ',    'EtFOSE',    'MeFOSE',    'FOSAA ', 'PF37DMOA ',
  'PF37DMOA',    '4_2FTS',    '6_2FTS',    '8_2FTS']
Length: 230, dtype: str

In [12]:
substance_before = set(gdf_merged.substance.unique())
gdf = gdf_merged.copy()

substance_rename = {
    '6_2-Cl-PFESA' : '6:2-Cl-PFESA',
    '8_2FTS' : "8:2 FTS", 
    '6_2FTS' : "6:2 FTS", 
    "4_2FTS" : "4:2 FTS"
}

gdf["substance"] = gdf["substance"].replace(substance_rename)
gdf["substance"] = gdf["substance"].str.strip()

gdf["substance"].unique()
substance_after = set(gdf.substance.unique())

In [13]:
matrix_rename = {
    "Wasser" : "surface water",
    'Sea' : "surface water",
    'Ground Water' : "groundwater",
    'River' : "surface water",
    'Tributary' : "surface water",
    'Wasser' : "surface water",
    'Estuary' : "surface water",
    'sea' : "surface water",
    'Lake' : "surface water",
    'canal' : "surface water",
    'Seawater' : "surface water",
    'Oppervlaktewater' : "surface water",
    'Pond' : "surface water",
    'Well' : "groundwater",
    'Groundwater' : "groundwater",
    'Spring water' : "groundwater",
    'Spring water 2 (feeding the cressionnières' : "groundwater",
    'Coastal seawater' : "surface water",
    'Port seawater' : "surface water",
    'Well ' : "groundwater",
    'groundwater' : "groundwater",
    'lake or river' : "surface water",
    'ditch (lake or river)' : "surface water",
    'recipient water (lake or river)' : "surface water",
    'storm water, lake or river' 'lake or river, coast' : "surface water",
    'Fire dam, lake or river' : "surface water",
    "storm water, lake or river" : "surface water",
    'lake or river, coast' : "surface water",
    "OW" : "surface water",
    "WASSER" : "surface water"
    
}
gdf["matrix"] = gdf["matrix"].replace(matrix_rename)

gdf = gdf[gdf.matrix == "surface water"]

In [14]:
gdf["year"] = gdf.date.dt.year
gdf_time = gdf[gdf.year > 2018]

gdf_time.lon = gdf_time.geometry.x
gdf_time.lat = gdf_time.geometry.y

In [15]:
gdf_time.dataset_name.unique()

gdf_time["category"] = "Sampling"
gdf_time["type"] = "Sampling location"

In [16]:
# Load PFAS Data Hub data and filter out unsuitable data
# df_meas = pd.read_parquet("../data/concentration_datasets/pdh_surface_water_sampling_relevant_parameters_2026-09-14.parquet")
df_meas = pd.read_parquet("../data/concentration_datasets/2026-09-14_pdh_surface_water_sampling_relevant_parameters.parquet")

# df_meas = pd.concat([df_meas, df_meas_])

gdf = gpd.GeoDataFrame(df_meas, geometry=gpd.points_from_xy(df_meas.lon, df_meas.lat), crs="EPSG:4326")
gdf = gdf[gdf["dataset_name"] != 'France - ICPE']
gdf = gdf[gdf["dataset_id"] != 101]
gdf = gdf[gdf["lat"] > 0]
gdf["date"] = pd.to_datetime(gdf["date"], format="ISO8601")


In [17]:
# Add BFG data 

df_bfg = gpd.read_file("../data/concentration_datasets/bfg_pfas.gpkg")
df_bfg["substance"] = df_bfg["parameter"]
df_bfg["conc"] = df_bfg["value"]
df_bfg = df_bfg.dropna(subset="substance")
df_bfg = df_bfg.to_crs(gdf.crs)
df_bfg["name"] = df_bfg["sampling_location"] + "_" + df_bfg["river"]
df_bfg["category"] = "Sampling"
df_bfg["type"] = "Sampling location"
df_bfg["source_type"] = "Authorities"
df_bfg["matrix"] = "Surface water"

df_bfg['year'] = df_bfg["date"].dt.year


df_bfg_single = df_bfg.drop_duplicates(subset=["geometry", "date", "substance", "conc"])

df_bfg_single["dataset_name"] = "_BfG"

cols = ['category', 'lat', 'lon', 'name', 'type',
       'source_type', 
       'dataset_name', 'matrix', 'date', 'year', 'substance',
       'unit', 'less_than', 'conc', 'geometry']
df_bfg_single = df_bfg_single[cols]

print(df_bfg_single.columns)

Index(['category', 'lat', 'lon', 'name', 'type', 'source_type', 'dataset_name',
       'matrix', 'date', 'year', 'substance', 'unit', 'less_than', 'conc',
       'geometry'],
      dtype='str')


In [18]:
df_full = pd.concat([gdf, df_bfg_single, gdf_time], join="outer")
df_full.loc[df_full["unit"] == "µg/L", "conc"] = df_full.loc[df_full["unit"] == "µg/L", "conc"] * 1000
df_full.loc[df_full["unit"] == "µg/L", "unit"] = "ng/l"
df_full.loc[df_full["unit"] == "µg/l", "conc"] = df_full.loc[df_full["unit"] == "µg/l", "conc"] * 1000
df_full.loc[df_full["unit"] == "µg/l", "unit"] = "ng/l"
df_full.loc[df_full["unit"] == "ug/l", "conc"] = df_full.loc[df_full["unit"] == "ug/l", "conc"] * 1000
df_full.loc[df_full["unit"] == "ug/l", "unit"] = "ng/l"

df_full.loc[df_full["unit"] == "ng/L", "unit"] = "ng/l"


substance_rename = {
    "Tetradecanoic acid, heptacosafluoro- (9CI)": "PFTeDA",
    "1-Heptanesulfonic acid, pentadecafluoro- (6CI)": "PFHpS",
    "PFU(n)DA" : "PFUnDA",
    "PFUnA" : "PFUnDA",
    "PFPA":"PFPeA",
    "PFTA" : "PFTeDA",
    "PFDoA" : "PFDoDA",
    "H4PFOS": "6:2 FTS",
    "6:2-FTS": "6:2 FTS",
    "NEtFBSAA":"N-EtFBSAA",
    "NEtFOSAA":"EtFOSAA",
    "NEtFPeSAA":"N-EtFPeSAA",
    "NMeFBSAA":"N-MeFBSAA",
    "NMeFHxSAA":"N-MeFHxSAA",
    "NMeFOSAA":"N-MeFOSAA",
    "NMeFPeSAA":"N-MeFPeSAA",
    "NMeFPrSAA":"N-MeFPrSAA",
    "PFOA_linear": "L_PFOA",
    "PFOS_linear": "L_PFOS",
    "PFHxS_linear":"L_PFHxS",
    "Ethanesulfonic acid, 2-[(6-chloro-1,1,2,2,3,3,4,4,5,5,6,6-dodecafluorohexyl)oxy]-1,1,2,2-tetrafluoro-, potassium salt (9CI)" : "F-53B",
    "H4PFDS":"8:2 FTS",
    "Perfluorhexansulfonsäure inkl. Isomere" : "PFHxS",
    "H4PFHxS" : "4:2 FTS",
    "N-Et-FOSA-A" : "EtFOSAA",
    "PFOSA" : "FOSA",
    "PFDoS" : "PFDoDS",
    "PFTrS" : "PFTrDS",
    "PfHxDA" : "PFHxDA",
}
df_full["substance"] = df_full["substance"].replace(substance_rename)

df_full["substance"].unique()


df_full = df_full[~df_full["substance"].isin(["Perfluorbutansulfonsäure inkl. Isomere","Perfluoroktansulfonsäure inkl. Isomere", "Perfluoroktansäure inkl. Isomere", "Summe aus 17 ausgewählten PFT", "Summe aus PFOA und PFOS", "Summe aus PFOA und PFOS inkl. Isomere", "som vertakte perfluorhexaansulfonzuur-isomeren", "som vertakte perfluoroctaansulfonzuur-isomeren"])]
df_full = df_full[~df_full["name"].isin(["LAUTERBOURG-KARLSRUHE_RHEIN", "Mannheim, Neckar", "Karlsruhe"])]


matrix_rename = {
    "Surface water": "surface water",
}
df_full["matrix"] = df_full["matrix"].replace(matrix_rename)
df_full = df_full[df_full["matrix"] == "surface water"]

In [19]:
# Remove any data point, that is not on land
hybas_mask = gpd.read_file("../data/auxiliary/hybas_eu_lev01_v1c.shp")
hybas_mask = hybas_mask.to_crs(gdf.crs)
df_full_hybas = gpd.clip(df_full, hybas_mask)

country_mask = gpd.read_file("../data/auxiliary/ne_10m_admin_0_countries.shp")
country_mask = country_mask.to_crs(gdf.crs)

df_full_country = gpd.clip(df_full, country_mask)

df_full = pd.concat([df_full_hybas, df_full_country])

# df_full.loc[df_full.substance == "ADONA", "substance"] = "DONA"
# df_full[df_full.substance == "DONA"]

df_full = df_full.drop_duplicates(subset=["geometry", "conc", "date", "substance"])

In [20]:
# Harmonise `less_than` field
df_full["less_than_"] = df_full["less_than"]
df_full["less_than"] = False
df_full.loc[df_full["less_than_"] == "True", "less_than"] = True
df_full.loc[df_full["less_than_"] == True, "less_than"] = True
df_full.loc[df_full["less_than_"] == "False", "less_than"] = False
df_full.loc[df_full["less_than_"] == False, "less_than"] = False

df_full = df_full.drop(columns="less_than_")

print(df_full["less_than"].unique())
print(df_full["less_than"].dtype)

[ True False]
bool


In [21]:
df_full = df_full[df_full.year > 2018]

print(len(df_full["substance"].unique()), "substances")
gdf_dropped = df_full[~df_full["less_than"]]

comps = df_full.dropna(subset="substance").copy()
comps["year"] = comps["year"].astype(int)
comps = comps[~comps["less_than"]] # This is important
counts = comps.groupby(["year", "substance"]).size().unstack(fill_value=0)

totals = counts.sum(axis=0)
# At least 200 samples above LOD for further analysis
selected_substances = totals[totals > 200].index.tolist()
selected_substances_by_lod = selected_substances

print(len(selected_substances)) # -1 for "Total", if total is included

219 substances
24


In [22]:

print(len(df_full["substance"].unique()))

comps = df_full.dropna(subset="substance").copy()
comps["year"] = comps["year"].astype(int)
counts = comps.groupby(["year", "substance"]).size().unstack(fill_value=0)

totals = counts.sum(axis=0)
# Select commonly monitored substances
selected_substances = totals[totals > 4000].index.tolist()
print(len(selected_substances)) # -1 for "Total", if total is included

219
36


In [23]:
selected_substances.extend(selected_substances_by_lod)
set_selected_substances = set(selected_substances)
selected_substances = sorted(list(set_selected_substances))

In [24]:
selected_substances

['4:2 FTS',
 '6:2 FTCA',
 '6:2 FTS',
 '8:2 FTS',
 'DONA',
 'EtFOSAA',
 'FOSA',
 'HFPO-DA',
 'L_PFOA',
 'L_PFOS',
 'N-Et-FOSA',
 'N-MeFOSAA',
 'PFBA',
 'PFBS',
 'PFDA',
 'PFDS',
 'PFDoDA',
 'PFDoDS',
 'PFHpA',
 'PFHpS',
 'PFHxA',
 'PFHxDA',
 'PFHxS',
 'PFNA',
 'PFNS',
 'PFOA',
 'PFODA',
 'PFOS',
 'PFPeA',
 'PFPeS',
 'PFTeDA',
 'PFTrDA',
 'PFTrDS',
 'PFUnDA',
 'PFUnDS',
 'TFA']

In [25]:
df_selected = df_full[df_full["substance"].isin(selected_substances)]
print(len(df_selected))
# Data cleaning
# print("Before REDACTED: ", len(df_selected))
df_selected = df_selected[df_selected.name != "REDACTED - IMPRECISE LOCATION"]
# print("Before Dates: ", len(df_selected))
df_selected = df_selected[~df_selected.date.isna()]
# print("Before Duplicates: ", len(df_selected))
df_selected = df_selected.drop_duplicates(subset=["geometry", "conc", "date", "substance"])
# print("Before Mainland EU: ", len(df_selected))
df_selected = df_selected[df_selected.lon > -30]
# print("Before concentration: ", len(df_selected))
df_selected = df_selected[df_selected["conc"]<10_000_000_000]
# print("Before concentration == 0: ", len(df_selected))
df_selected = df_selected[df_selected.conc != 0]
# print("Before units: ",len(df_selected))
df_selected = df_selected[df_selected["unit"] != 'ng/kg fw']
df_selected = df_selected[df_selected["unit"] != 'ng/kg']
df_selected = df_selected[df_selected["unit"] != '']
# print("Before CASID: ",len(df_selected))


print(len(df_selected))

1946992
1941897


In [26]:
# Attach country names

europe = country_mask[country_mask["CONTINENT"].isin(["Europe", "Asia"])]
europe = europe.to_crs(df_selected.crs)

europe_no_russia = europe[
    europe["ADMIN"] != "Russia"
]  # Done due to wrong assignment. There are no sammpling stations in Russia in this dataset

within = gpd.sjoin(
    df_selected.to_crs("EPSG:3035"),
    europe_no_russia[["ADMIN", "geometry"]].to_crs("EPSG:3035"),
    how="left",
    predicate="within",
)

missing = within[within["ADMIN"].isna()].drop(columns="ADMIN")

if "index_right" in missing.columns:
    missing = missing.drop(columns="index_right")

nearest = gpd.sjoin_nearest(
    missing.to_crs("EPSG:3035"),
    europe_no_russia[["ADMIN", "geometry"]].to_crs("EPSG:3035"),
    how="left",
)

# combine back
df_selected = pd.concat([within[within["ADMIN"].notna()], nearest])
df_selected["country"] = df_selected["ADMIN"]

In [27]:
cols_to_keep = ['lat', 'lon', 'name', 'country',
       'source_type', 'data_collection_method', 'source_text', 'source_url',
       'dataset_id', 'dataset_name', 'matrix', 'date', 'year', 'geometry',
       'substance', 'unit', 'less_than', 'cas_id', 'conc']
df_selected = df_selected[cols_to_keep]

In [28]:
df_selected.dtypes

lat                              float64
lon                              float64
name                                 str
country                              str
source_type                          str
data_collection_method               str
source_text                          str
source_url                           str
dataset_id                       float64
dataset_name                         str
matrix                               str
date                      datetime64[ns]
year                             float64
geometry                        geometry
substance                            str
unit                                 str
less_than                           bool
cas_id                               str
conc                             float64
dtype: object

In [29]:
basins_lev04 = gpd.read_file("../data/auxiliary/hybas_eu_lev04_v1c.shp")


In [30]:
from pfas_assessment_europe.constants import HYBAS_RIVER_RENAME

# Attach HYBASlev04 (basins)

id_name = "HYBAS_ID"

within = gpd.sjoin(
    df_selected.to_crs("EPSG:3035"),
    basins_lev04[[id_name, "geometry"]].to_crs("EPSG:3035"),
    how="left",
    predicate="within",
)

missing = within[within[id_name].isna()].drop(columns=id_name)

if "index_right" in missing.columns:
    missing = missing.drop(columns="index_right")

nearest = gpd.sjoin_nearest(
    missing.to_crs("EPSG:3035"),
    basins_lev04[[id_name, "geometry"]].to_crs("EPSG:3035"),
    how="left",
)

# Combine back
gdf_joined = pd.concat([within[within[id_name].notna()], nearest])
gdf_joined["basin"] = gdf_joined[id_name]


In [31]:
gdf_joined_bas = gdf_joined.copy()
gdf_joined_bas["basin"] = gdf_joined_bas["basin"].replace(HYBAS_RIVER_RENAME)
gdf_joined_bas["basin"] = gdf_joined_bas["basin"].astype("string")
gdf_joined_bas["HYBAS_ID04"] = gdf_joined_bas["HYBAS_ID"].copy()
gdf_joined_bas = gdf_joined_bas.drop(columns=["index_right", "HYBAS_ID"])

In [32]:
print(len(df_selected))
print(len(gdf_joined_bas))


1941897
1941897


In [33]:
# df_selected = df_selected.to_crs("EPSG:4326")
# df_selected.to_file("../data/input/pfas_data.gpkg", driver="GPKG") # Save to data/input to be used in the Analysis
gdf_joined_bas = gdf_joined_bas.to_crs("EPSG:4326")
gdf_joined_bas.reset_index(drop=True).to_parquet("../data/input/pfas_data.parquet")

In [34]:
gdf_joined_bas.dtypes

lat                              float64
lon                              float64
name                                 str
country                              str
source_type                          str
data_collection_method               str
source_text                          str
source_url                           str
dataset_id                       float64
dataset_name                         str
matrix                               str
date                      datetime64[ns]
year                             float64
geometry                        geometry
substance                            str
unit                                 str
less_than                           bool
cas_id                               str
conc                             float64
basin                             string
HYBAS_ID04                       float64
dtype: object

In [35]:
df_selected.substance.unique()

<ArrowStringArray>
[     'PFOS',      'PFOA',      'PFBA',     'PFPeA',    'PFDoDA',     'PFHpA',
      'PFDS',     'PFHxA',     'PFHxS',      'PFNA',      'PFDA',     'PFHpS',
    'PFUnDS',    'PFDoDS',      'PFNS',    'PFTrDS',     'PFPeS',    'PFTrDA',
    'PFUnDA',      'PFBS',      'FOSA',     'PFODA',    'PFHxDA',   '6:2 FTS',
 'N-MeFOSAA',   'EtFOSAA',   '4:2 FTS',   'HFPO-DA',  '6:2 FTCA',    'PFTeDA',
   '8:2 FTS',       'TFA', 'N-Et-FOSA',      'DONA',    'L_PFOS',    'L_PFOA']
Length: 36, dtype: str

In [36]:
df_selected.unit.unique()

<ArrowStringArray>
['ng/l']
Length: 1, dtype: str

In [37]:
len(df_selected)

1941897